In [1]:
import os
import json
import re
import copy
import warnings
from typing import Dict, Any, Tuple, Optional, Union
import sys
import pandas as pd
from music21 import converter, stream, clef, key, meter, instrument

from muster import muster

sys.path.append('/home/hbli/PM2S/MIDI2ScoreTransformer/midi2scoretransformer/')
from utils import score_similarity_normalized

/home/hbli/songformer/env/miniforge3/envs/midi2scoretf/lib/python3.11/site-packages/pretty_midi/instrument.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [27]:
def _as_score(x):
    """
    x 可以是:
    - path(str)
    - music21.stream.Score
    """
    if isinstance(x, str):
        return converter.parse(x)
    return x


def _deepcopy_score(score):
    # 某些情况下直接用原对象也行，但 case study 中建议 copy 一份更稳
    return copy.deepcopy(score)


def parse_half_open_measure_range(s: str):
    """
    '[10:20]' -> (10, 20)
    表示第10到第19小节（右开）
    """
    s = s.replace(" ", "")
    m = re.match(r"\[(\d+):(\d+)\]", s)
    if m is None:
        raise ValueError(f"Bad range format: {s}, expected like '[10:20]'")
    start = int(m.group(1))
    end_exclusive = int(m.group(2))
    if end_exclusive <= start:
        raise ValueError(f"Need end > start, got {s}")
    return start, end_exclusive


def get_max_measure_number(score):
    """
    取各 part 中最大的 measure number
    """
    max_num = 0
    for part in score.parts:
        for m in part.getElementsByClass(stream.Measure):
            if m.number is not None:
                max_num = max(max_num, int(m.number))
    return max_num


def range_covers_full_score(score, start: int, end_exclusive: int):
    """
    判断传入范围是否已经覆盖整首
    """
    max_measure = get_max_measure_number(score)
    return start <= 1 and end_exclusive > max_measure


def _inject_context_into_first_measure(orig_part, frag_part, start_measure: int):
    """
    给 fragment 首小节补齐 clef/key/time 上下文
    """
    first_measure = frag_part.measure(start_measure)
    orig_measure = orig_part.measure(start_measure)

    if first_measure is None or orig_measure is None:
        return frag_part

    for cls in (clef.Clef, key.KeySignature, meter.TimeSignature):
        if len(first_measure.getElementsByClass(cls)) == 0:
            ctx = orig_measure.getContextByClass(cls)
            if ctx is not None:
                first_measure.insert(0, copy.deepcopy(ctx))

    return frag_part


def slice_score_measures(score, start: int, end_exclusive: int):
    """
    从整首 score 中裁 [start, end_exclusive) 小节。
    返回一个新的 music21 Score 对象。
    """
    score = _as_score(score)

    # 如果其实要的是整首，直接 deep copy 整首，避免重构路径
    if range_covers_full_score(score, start, end_exclusive):
        return _deepcopy_score(score)

    end_inclusive = end_exclusive - 1
    frag_score = stream.Score()

    if score.metadata is not None:
        frag_score.metadata = copy.deepcopy(score.metadata)

    # 尽量保留 score-level 元素（如部分 layout / metadata 之外的信息）
    for el in score:
        if not isinstance(el, stream.Part):
            try:
                frag_score.insert(el.offset, copy.deepcopy(el))
            except Exception:
                pass

    for part in score.parts:
        frag_part = part.measures(start, end_inclusive)
        if frag_part is None:
            continue

        # instrument 尽量补上
        insts = part.getElementsByClass(instrument.Instrument)
        if insts and len(frag_part.getElementsByClass(instrument.Instrument)) == 0:
            frag_part.insert(0, copy.deepcopy(insts[0]))

        frag_part = _inject_context_into_first_measure(part, frag_part, start)
        frag_score.insert(0, frag_part)

    return frag_score


def flatten_metrics(sim):
    out = {}

    if sim is None:
        return out

    if "muster" in sim and isinstance(sim["muster"], dict):
        for k, v in sim["muster"].items():
            out[f"muster.{k}"] = v

    if "mxl <-> gt_mxl" in sim and isinstance(sim["mxl <-> gt_mxl"], dict):
        for k, v in sim["mxl <-> gt_mxl"].items():
            out[f"mxl.{k}"] = v

    return out


# ========= 核心：局部评估 =========

def compare_measure_range(
    est_score_or_path,
    gt_score_or_path,
    measure_range: str,
    full: bool = False,
    return_fragments: bool = False,
):
    """
    例子:
    compare_measure_range(path1, path2, '[10:20]')

    含义:
    - 读取两首完整 score
    - 裁出第10到第19小节
    - 直接调用 muster / score_similarity_normalized 评估这段
    """
    start, end_exclusive = parse_half_open_measure_range(measure_range)

    est_full = _as_score(est_score_or_path)
    gt_full = _as_score(gt_score_or_path)

    est_frag = slice_score_measures(est_full, start, end_exclusive)
    gt_frag = slice_score_measures(gt_full, start, end_exclusive)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        sim = {
            "muster": muster(est_frag, gt_frag),
            "mxl <-> gt_mxl": score_similarity_normalized(est_frag, gt_frag, full=full),
        }

    flat = flatten_metrics(sim)
    flat["measure_range"] = measure_range
    flat["start_measure"] = start
    flat["end_exclusive"] = end_exclusive
    flat["end_measure"] = end_exclusive - 1

    row = pd.Series(flat)

    if return_fragments:
        return row, est_frag, gt_frag
    return row


# ========= 滑窗扫描 =========

def scan_measure_windows(
    est_score_or_path,
    gt_score_or_path,
    start_measure: int,
    end_exclusive: int,
    window: int = 4,
    step: int = 1,
    full: bool = False,
):
    """
    扫一串窗口:
    [1:5], [2:6], [3:7]...
    """
    rows = []
    est_full = _as_score(est_score_or_path)
    gt_full = _as_score(gt_score_or_path)

    for s in range(start_measure, end_exclusive - window + 1, step):
        e = s + window
        r = f"[{s}:{e}]"
        row = compare_measure_range(est_full, gt_full, r, full=full, return_fragments=False)
        rows.append(row)

    if len(rows) == 0:
        return pd.DataFrame()

    return pd.DataFrame(rows)


# ========= 辅助：看某首的整首范围 =========

def compare_full_song(est_score_or_path, gt_score_or_path, full: bool = False):
    est_full = _as_score(est_score_or_path)
    gt_full = _as_score(gt_score_or_path)

    max_measure = max(get_max_measure_number(est_full), get_max_measure_number(gt_full))
    return compare_measure_range(est_full, gt_full, f"[1:{max_measure + 1}]", full=full)

In [31]:
path1 = '/mnt/ssd/hbli/midi2scoretransformer/runs/best_ckpt_preds/test/preds/Ravel/Gaspard_de_la_Nuit/1_Ondine/pred_score.musicxml'
path2 = '/mnt/ssd/hbli/datasets/PM2S_dataset/midi2scoretransformer/asap-dataset/Ravel/Gaspard_de_la_Nuit/1_Ondine/xml_score.musicxml'
compare_measure_range(path1, path2, '[1:10]')

muster.PitchER         0.974026
muster.MissRate         21.4286
muster.ExtraRate        22.4359
muster.OnsetER           22.314
muster.OffsetER         50.8264
muster.MeanER           23.5958
muster.VoiceER          55.3719
muster.NewMeanER        28.8918
muster.Pvoi             50.8511
muster.Rvoi             31.3889
muster.Fvoi             38.8171
muster.ScaleErr         1.52769
muster.HandER           51.2397
mxl.Clef                    0.0
mxl.KeySignature            0.0
mxl.TimeSignature           0.0
mxl.NoteDeletion       0.595082
mxl.NoteInsertion      0.103279
mxl.NoteSpelling       0.336066
mxl.NoteDuration       0.321311
mxl.StemDirection      0.072131
mxl.Beams              0.211475
mxl.Tie                     0.0
mxl.StaffAssignment        None
mxl.Voice                   0.0
mxl.TrillTP                   0
mxl.TrillFP                   0
mxl.TrillTN                 247
mxl.TrillFN                   0
mxl.StaccatoTP                0
mxl.StaccatoFP                0
mxl.Stac

In [2]:
df_metrics = pd.read_excel('/mnt/ssd/hbli/midi2scoretransformer/runs/best_ckpt_preds/test/metrics/eval_test.xlsx', sheet_name='per_song')

In [6]:
df_metrics['song_name'] = df_metrics['relpath'].apply(lambda x: os.path.dirname(x))

In [3]:
df_metrics.sort_values(by=['muster.PitchER', 'muster.MissRate'], ascending=False)

,idx,relpath,midi_path,gt_xml_path,pred_xml_path,n_notes,composer,piece,subpath,muster.PitchER,...,mxl.TrillPrec,mxl.TrillRec,mxl.TrillF1,mxl.StaccatoPrec,mxl.StaccatoRec,mxl.StaccatoF1,mxl.GracePrec,mxl.GraceRec,mxl.GraceF1,mxl.n_Note
27,27,Ravel/Gaspard_de_la_Nuit/1_Ondine/KotysV11.mid,/mnt/ssd/hbli/datasets/PM2S_dataset/midi2score...,/mnt/ssd/hbli/datasets/PM2S_dataset/midi2score...,/mnt/ssd/hbli/midi2scoretransformer/runs/best_...,4514,Ravel,Gaspard_de_la_Nuit,1_Ondine/KotysV11.mid,10.480100,...,NaN,NaN,NaN,0.014458,1.000000,0.028504,0.000000,0.000000,NaN,5066
35,35,Prokofiev/Toccata/Kociuban16.mid,/mnt/ssd/hbli/datasets/PM2S_dataset/midi2score...,/mnt/ssd/hbli/datasets/PM2S_dataset/midi2score...,/mnt/ssd/hbli/midi2scoretransformer/runs/best_...,4843,Prokofiev,Toccata,Kociuban16.mid,8.354980,...,NaN,0.000000,NaN,0.017036,0.961538,0.033478,0.000000,0.000000,NaN,4656
24,24,Ravel/Gaspard_de_la_Nuit/1_Ondine/Albright01.mid,/mnt/ssd/hbli/datasets/PM2S_dataset/midi2score...,/mnt/ssd/hbli/datasets/PM2S_dataset/midi2score...,/mnt/ssd/hbli/midi2scoretransformer/runs/best_...,4422,Ravel,Gaspard_de_la_Nuit,1_Ondine/Albright01.mid,7.194680,...,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,0.000000,NaN,5066
29,29,Ravel/Gaspard_de_la_Nuit/1_Ondine/Ladid03.mid,/mnt/ssd/hbli/datasets/PM2S_dataset/midi2score...,/mnt/ssd/hbli/datasets/PM2S_dataset/midi2score...,/mnt/ssd/hbli/midi2scoretransformer/runs/best_...,4573,Ravel,Gaspard_de_la_Nuit,1_Ondine/Ladid03.mid,6.477000,...,NaN,NaN,NaN,0.000000,0.000000,NaN,1.000000,0.064516,0.121212,5066
39,39,Schubert/Impromptu_op.90_D.899/1/Duepree08M.mid,/mnt/ssd/hbli/datasets/PM2S_dataset/midi2score...,/mnt/ssd/hbli/datasets/PM2S_dataset/midi2score...,/mnt/ssd/hbli/midi2scoretransformer/runs/best_...,4968,Schubert,Impromptu_op.90_D.899,1/Duepree08M.mid,6.381120,...,NaN,0.000000,NaN,0.234127,0.307292,0.265766,0.571429,0.500000,0.533333,3789
36,36,Schubert/Impromptu_op.90_D.899/1/Teo06M.mid,/mnt/ssd/hbli/datasets/PM2S_dataset/midi2score...,/mnt/ssd/hbli/datasets/PM2S_dataset/midi2score...,/mnt/ssd/hbli/midi2scoretransformer/runs/best_...,4850,Schubert,Impromptu_op.90_D.899,1/Teo06M.mid,6.264570,...,NaN,0.000000,NaN,0.089776,0.195652,0.123077,0.285714,0.500000,0.363636,3789
58,58,Prokofiev/Toccata/Dossin07.mid,/mnt/ssd/hbli/datasets/PM2S_dataset/midi2score...,/mnt/ssd/hbli/datasets/PM2S_dataset/midi2score...,/mnt/ssd/hbli/midi2scoretransformer/runs/best_...,5204,Prokofiev,Toccata,Dossin07.mid,6.191820,...,NaN,0.000000,NaN,0.015208,1.000000,0.029960,0.000000,0.000000,NaN,4656
13,13,Debussy/Images_Book_1/1_Reflets_dans_lEau/Klei...,/mnt/ssd/hbli/datasets/PM2S_dataset/midi2score...,/mnt/ssd/hbli/datasets/PM2S_dataset/midi2score...,/mnt/ssd/hbli/midi2scoretransformer/runs/best_...,2019,Debussy,Images_Book_1,1_Reflets_dans_lEau/Kleisen11M.mid,6.057650,...,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,2073
23,23,Ravel/Gaspard_de_la_Nuit/1_Ondine/GarritsonL05...,/mnt/ssd/hbli/datasets/PM2S_dataset/midi2score...,/mnt/ssd/hbli/datasets/PM2S_dataset/midi2score...,/mnt/ssd/hbli/midi2scoretransformer/runs/best_...,4323,Ravel,Gaspard_de_la_Nuit,1_Ondine/GarritsonL05.mid,5.965340,...,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,0.000000,NaN,5066
25,25,Prokofiev/Toccata/Teo05.mid,/mnt/ssd/hbli/datasets/PM2S_dataset/midi2score...,/mnt/ssd/hbli/datasets/PM2S_dataset/midi2score...,/mnt/ssd/hbli/midi2scoretransformer/runs/best_...,4453,Prokofiev,Toccata,Teo05.mid,5.916650,...,NaN,0.000000,NaN,0.028571,0.836364,0.055255,NaN,0.000000,NaN,4656


In [13]:
df_metrics.groupby('song_name')[['muster.PitchER', 'muster.MissRate', 'muster.ExtraRate', 'muster.OnsetER', 'muster.OffsetER', 'mxl.NoteSpelling', 'mxl.NoteDuration']].mean()

,muster.PitchER,muster.MissRate,muster.ExtraRate,muster.OnsetER,muster.OffsetER,mxl.NoteSpelling,mxl.NoteDuration
song_name,,,,,,,
Bach/Fugue/bwv_846,0.671141,3.758390,2.448980,3.626220,17.573200,0.000000,0.804000
Beethoven/Piano_Sonatas/10-1,1.197370,6.913870,3.018110,12.074700,20.912900,0.010943,0.244906
Brahms/Six_Pieces_op_118/2,0.064020,2.112680,8.278340,2.812300,19.882300,0.014229,0.618668
Chopin/Ballades/1,2.253176,4.492297,2.355028,11.419618,19.045565,0.034344,0.810316
Debussy/Images_Book_1/1_Reflets_dans_lEau,4.059755,7.855735,5.386405,26.715950,30.276500,0.089243,0.573324
Haydn/Keyboard_Sonatas/31-1,0.644940,2.414425,3.641655,10.131647,16.763175,0.004299,0.356316
Liszt/Annees_de_pelerinage_2/1_Gondoliera,3.029170,14.603550,6.933965,16.698200,26.965750,0.006525,0.387211
Mozart/Piano_Sonatas/12-1,0.426483,2.223800,2.044128,5.928155,19.606200,0.007511,0.688388
Prokofiev/Toccata,5.328550,8.450334,3.198854,1.986660,15.830187,0.090985,0.156545


In [4]:
df_metrics.columns

Index(['idx', 'relpath', 'midi_path', 'gt_xml_path', 'pred_xml_path',
       'n_notes', 'composer', 'piece', 'subpath', 'muster.PitchER',
       'muster.MissRate', 'muster.ExtraRate', 'muster.OnsetER',
       'muster.OffsetER', 'muster.MeanER', 'muster.VoiceER',
       'muster.NewMeanER', 'muster.Pvoi', 'muster.Rvoi', 'muster.Fvoi',
       'muster.ScaleErr', 'muster.HandER', 'mxl.Clef', 'mxl.KeySignature',
       'mxl.TimeSignature', 'mxl.NoteDeletion', 'mxl.NoteInsertion',
       'mxl.NoteSpelling', 'mxl.NoteDuration', 'mxl.StemDirection',
       'mxl.Beams', 'mxl.Tie', 'mxl.StaffAssignment', 'mxl.Voice',
       'mxl.TrillTP', 'mxl.TrillFP', 'mxl.TrillTN', 'mxl.TrillFN',
       'mxl.StaccatoTP', 'mxl.StaccatoFP', 'mxl.StaccatoTN', 'mxl.StaccatoFN',
       'mxl.GraceTP', 'mxl.GraceFP', 'mxl.GraceTN', 'mxl.GraceFN',
       'mxl.TrillPrec', 'mxl.TrillRec', 'mxl.TrillF1', 'mxl.StaccatoPrec',
       'mxl.StaccatoRec', 'mxl.StaccatoF1', 'mxl.GracePrec', 'mxl.GraceRec',
       'mxl.GraceF1',

In [23]:
df_metrics['mxl.NoteDuration'].mean()

0.5583470946604876

In [22]:
df_metrics.sort_values(by=['muster.PitchER', 'muster.MissRate'], ascending=False)[['relpath', 'muster.PitchER',
       'muster.MissRate', 'muster.ExtraRate', 'muster.OnsetER', 'muster.OffsetER', 'mxl.NoteSpelling', 'mxl.NoteDuration']]

,relpath,muster.PitchER,muster.MissRate,muster.ExtraRate,muster.OnsetER,muster.OffsetER,mxl.NoteSpelling,mxl.NoteDuration
27,Ravel/Gaspard_de_la_Nuit/1_Ondine/KotysV11.mid,10.480100,27.53830,17.91250,49.50660,39.7801,0.465061,0.613897
35,Prokofiev/Toccata/Kociuban16.mid,8.354980,7.16450,3.22653,2.40149,12.8934,0.089991,0.087629
24,Ravel/Gaspard_de_la_Nuit/1_Ondine/Albright01.mid,7.194680,23.61950,11.17880,41.18730,34.7493,0.449862,0.605606
29,Ravel/Gaspard_de_la_Nuit/1_Ondine/Ladid03.mid,6.477000,18.88620,8.98800,38.08460,33.4080,0.484406,0.678642
39,Schubert/Impromptu_op.90_D.899/1/Duepree08M.mid,6.381120,9.03263,36.55760,14.99040,27.6105,0.006334,0.583795
36,Schubert/Impromptu_op.90_D.899/1/Teo06M.mid,6.264570,11.01400,36.59950,15.06220,26.4571,0.005278,0.385590
58,Prokofiev/Toccata/Dossin07.mid,6.191820,6.14852,4.93421,1.70704,15.1557,0.086340,0.110180
13,Debussy/Images_Book_1/1_Reflets_dans_lEau/Klei...,6.057650,9.57499,6.42063,30.52400,32.4149,0.086831,0.512783
23,Ravel/Gaspard_de_la_Nuit/1_Ondine/GarritsonL05...,5.965340,21.72510,6.90316,37.53860,31.8744,0.456771,0.524872
25,Prokofiev/Toccata/Teo05.mid,5.916650,15.89290,2.67366,1.87420,20.2311,0.078823,0.123497


### Case Study

#### if midi metadata have (correct) time signature or key signature

In [11]:
data_dir = "/mnt/ssd/hbli/datasets/PM2S_dataset/midi2scoretransformer/"
asap_annotations = json.load(
    open(data_dir + "/asap-dataset/asap_annotations.json")
)

In [15]:
for k, v in asap_annotations.items():
    if 'Brahms' in k:
        print(k)

Brahms/Six_Pieces_op_118/2/Shilyaev03.mid


In [17]:
brahms = asap_annotations['Brahms/Six_Pieces_op_118/2/Shilyaev03.mid']

In [18]:
for k, v in brahms.items():
    print(k)

performance_beats
performance_downbeats
performance_beats_type
perf_time_signatures
perf_key_signatures
midi_score_beats
midi_score_downbeats
midi_score_beats_type
midi_score_time_signatures
midi_score_key_signatures
downbeats_score_map
score_and_performance_aligned


In [21]:
brahms['perf_time_signatures'], brahms['perf_key_signatures']

({'2.859239': ['3/4', 3]}, {'1.6272685': [9, 3]})